# COPY INTO - Efficient Data Ingestion

**COPY INTO** is a SQL command in Databricks for efficiently loading data from cloud storage into Delta tables. It provides idempotent, incremental data loading with built-in file tracking.

## Key Features

✅ **Idempotent** — Same file loaded multiple times = loaded once  
✅ **Incremental** — Automatically tracks processed files  
✅ **File format support** — CSV, JSON, Parquet, Avro, ORC, Text  
✅ **Schema enforcement** — Validates data against table schema  
✅ **Error handling** — Skip bad records or fail fast  
✅ **Simple syntax** — No complex streaming code needed  

---

## When to Use COPY INTO

### ✅ Good Use Cases:
* **Batch file ingestion** — Process files on schedule (hourly, daily)
* **Small to medium files** — Up to thousands of files
* **Idempotent loads** — Need to avoid duplicate loading
* **Simple ETL** — Straightforward file-to-table loading
* **Schema enforcement** — Validate incoming data structure

### ⚠️ Consider Alternatives For:
* **Large-scale ingestion** → Use **Auto Loader** (cloudFiles)
* **Real-time streaming** → Use **Structured Streaming**
* **Millions of files** → Use **Auto Loader**
* **Complex transformations** → Use **Spark DataFrames** or **Lakeflow Pipelines**

---

## Basic Syntax

```sql
COPY INTO target_table
FROM 'source_path'
FILEFORMAT = format
FORMAT_OPTIONS ('option1' = 'value1', ...)
COPY_OPTIONS ('option1' = 'value1', ...)
```

---

## COPY INTO vs Other Methods

| Method | Files Tracked | Idempotent | Streaming | Scale | Use Case |
|--------|---------------|------------|-----------|-------|----------|
| **COPY INTO** | ✅ Yes | ✅ Yes | ❌ No | Medium | Batch file loads |
| **Auto Loader** | ✅ Yes | ✅ Yes | ✅ Yes | Large | Production streaming |
| **Structured Streaming** | ❌ No* | ❌ No* | ✅ Yes | Large | Real-time processing |
| **INSERT INTO** | ❌ No | ❌ No | ❌ No | Any | Manual loads |

*Unless using checkpoint locations

---

Let's explore with hands-on examples!

In [0]:
# Create sample data files for COPY INTO demonstrations
import json
import pandas as pd

# Define base path using Unity Catalog Volume
# Note: /tmp/ and workspace paths are not writable on serverless compute
base_path = "/Volumes/workspace/default/copy_into_demo"

# Clean up any existing data (suppress error if doesn't exist)
try:
    dbutils.fs.rm(base_path, recurse=True)
except:
    pass  # Ignore if path doesn't exist

# Create directories
dbutils.fs.mkdirs(f"{base_path}/csv")
dbutils.fs.mkdirs(f"{base_path}/json")
dbutils.fs.mkdirs(f"{base_path}/parquet")

print(f"Created directories under: {base_path}")

# 1. Create CSV files
csv_data_1 = """id,name,age,city,salary
1,Alice,30,New York,75000
2,Bob,25,San Francisco,65000
3,Charlie,35,Seattle,80000"""

csv_data_2 = """id,name,age,city,salary
4,Diana,28,Boston,70000
5,Eve,32,Austin,72000
6,Frank,29,Denver,68000"""

dbutils.fs.put(f"{base_path}/csv/employees_1.csv", csv_data_1, overwrite=True)
dbutils.fs.put(f"{base_path}/csv/employees_2.csv", csv_data_2, overwrite=True)

print("\n✅ Created CSV files:")
print(f"   - {base_path}/csv/employees_1.csv (3 rows)")
print(f"   - {base_path}/csv/employees_2.csv (3 rows)")

# 2. Create JSON files
json_data_1 = [{"id": 101, "product": "Laptop", "price": 1200.50, "category": "Electronics"},
               {"id": 102, "product": "Mouse", "price": 25.99, "category": "Electronics"},
               {"id": 103, "product": "Desk", "price": 350.00, "category": "Furniture"}]

json_data_2 = [{"id": 104, "product": "Chair", "price": 200.00, "category": "Furniture"},
               {"id": 105, "product": "Monitor", "price": 450.00, "category": "Electronics"}]

json_str_1 = "\n".join([json.dumps(record) for record in json_data_1])
json_str_2 = "\n".join([json.dumps(record) for record in json_data_2])

dbutils.fs.put(f"{base_path}/json/products_1.json", json_str_1, overwrite=True)
dbutils.fs.put(f"{base_path}/json/products_2.json", json_str_2, overwrite=True)

print("\n✅ Created JSON files:")
print(f"   - {base_path}/json/products_1.json (3 rows)")
print(f"   - {base_path}/json/products_2.json (2 rows)")

# 3. Create Parquet files
df_orders_1 = pd.DataFrame({
    'order_id': [1001, 1002, 1003],
    'customer_id': [1, 2, 3],
    'amount': [150.00, 200.00, 175.00],
    'order_date': ['2026-06-01', '2026-06-02', '2026-06-03']
})

df_orders_2 = pd.DataFrame({
    'order_id': [1004, 1005],
    'customer_id': [4, 5],
    'amount': [225.00, 190.00],
    'order_date': ['2026-06-04', '2026-06-05']
})

spark.createDataFrame(df_orders_1).write.mode("overwrite").parquet(f"{base_path}/parquet/orders_1.parquet")
spark.createDataFrame(df_orders_2).write.mode("overwrite").parquet(f"{base_path}/parquet/orders_2.parquet")

print("\n✅ Created Parquet files:")
print(f"   - {base_path}/parquet/orders_1.parquet (3 rows)")
print(f"   - {base_path}/parquet/orders_2.parquet (2 rows)")

print("\n" + "="*60)
print("✅ Sample data files created successfully!")
print("="*60)

In [0]:
%sql
-- Basic COPY INTO example with CSV files
-- Step 1: Create target Delta table

CREATE OR REPLACE TABLE employees (
  id INT,
  name STRING,
  age INT,
  city STRING,
  salary DOUBLE
);

-- Step 2: COPY data from CSV files
COPY INTO employees
FROM '/Volumes/workspace/default/copy_into_demo/csv/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
COPY_OPTIONS ('mergeSchema' = 'false');

-- Step 3: Verify the data was loaded
SELECT * FROM employees ORDER BY id;

In [0]:
%sql
-- Demonstrate idempotency: running COPY INTO again won't duplicate data
-- This is a key feature of COPY INTO!

SELECT 'Before re-running COPY INTO' as stage, COUNT(*) as row_count FROM employees;

-- Run COPY INTO again with the same files
COPY INTO employees
FROM '/Volumes/workspace/default/copy_into_demo/csv/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false');

SELECT 'After re-running COPY INTO' as stage, COUNT(*) as row_count FROM employees;

-- ✅ Notice: Row count stays the same!
-- COPY INTO tracks which files have been processed and skips them

In [0]:
%sql
-- COPY INTO with JSON files
-- Step 1: Create target table for products

CREATE OR REPLACE TABLE products (
  id INT,
  product STRING,
  price DOUBLE,
  category STRING
);

-- Step 2: COPY from JSON files
COPY INTO products
FROM '/Volumes/workspace/default/copy_into_demo/json/'
FILEFORMAT = JSON
COPY_OPTIONS ('mergeSchema' = 'false');

-- Step 3: Verify data
SELECT * FROM products ORDER BY id;

-- View load statistics
SELECT 
  category,
  COUNT(*) as product_count,
  AVG(price) as avg_price,
  MIN(price) as min_price,
  MAX(price) as max_price
FROM products
GROUP BY category
ORDER BY category;

In [0]:
%sql
-- COPY INTO with Parquet files
-- Parquet is a columnar format - very efficient!

-- Step 1: Create target table
CREATE OR REPLACE TABLE orders (
  order_id INT,
  customer_id INT,
  amount DOUBLE,
  order_date DATE
);

-- Step 2: COPY from Parquet files
COPY INTO orders
FROM '/Volumes/workspace/default/copy_into_demo/parquet/'
FILEFORMAT = PARQUET;

-- Step 3: Query the data
SELECT * FROM orders ORDER BY order_date;

-- Calculate summary statistics
SELECT 
  COUNT(*) as total_orders,
  SUM(amount) as total_revenue,
  AVG(amount) as avg_order_value,
  MIN(order_date) as first_order,
  MAX(order_date) as last_order
FROM orders;

In [0]:
%sql
-- Load specific files using pattern matching
-- Useful when you want to load only certain files from a directory

CREATE OR REPLACE TABLE employees_subset (
  id INT,
  name STRING,
  age INT,
  city STRING,
  salary DOUBLE
);

-- Load only files matching a pattern (e.g., only employees_1.csv)
COPY INTO employees_subset
FROM '/Volumes/workspace/default/copy_into_demo/csv/'
FILEFORMAT = CSV
PATTERN = 'employees_1.csv'
FORMAT_OPTIONS ('header' = 'true');

SELECT 'Only employees_1.csv loaded' as note, COUNT(*) as row_count FROM employees_subset;

-- You can also use wildcards
-- PATTERN = '*.csv'         -- All CSV files
-- PATTERN = 'employees_*.csv' -- Files starting with employees_
-- PATTERN = '*_1.csv'        -- Files ending with _1.csv

In [0]:
%sql
-- Advanced FORMAT_OPTIONS for CSV files
-- Handle different delimiters, quotes, escape characters, etc.

CREATE OR REPLACE TABLE employees_advanced (
  id INT,
  name STRING,
  age INT,
  city STRING,
  salary DOUBLE
);

-- COPY with various CSV format options
COPY INTO employees_advanced
FROM '/Volumes/workspace/default/copy_into_demo/csv/'
FILEFORMAT = CSV
FORMAT_OPTIONS (
  'header' = 'true',           -- First row is header
  'delimiter' = ',',           -- Column delimiter (default: comma)
  'quote' = '"',               -- Quote character for strings
  'escape' = '\\',             -- Escape character
  'nullValue' = 'NULL',        -- String to interpret as NULL
  'emptyValue' = '',           -- How to handle empty strings
  'dateFormat' = 'yyyy-MM-dd', -- Date format pattern
  'timestampFormat' = 'yyyy-MM-dd HH:mm:ss', -- Timestamp format
  'inferSchema' = 'false',     -- Don't infer schema (use table schema)
  'mode' = 'PERMISSIVE'        -- PERMISSIVE, DROPMALFORMED, or FAILFAST
);

SELECT * FROM employees_advanced ORDER BY id;

-- Common FORMAT_OPTIONS:
-- CSV: header, delimiter, quote, escape, nullValue, dateFormat
-- JSON: dateFormat, timestampFormat, multiLine
-- All: mode (PERMISSIVE/DROPMALFORMED/FAILFAST)

In [0]:
%sql
-- COPY_OPTIONS control the copy behavior

-- Option 1: Force reload (ignore file tracking)
CREATE OR REPLACE TABLE employees_force (
  id INT,
  name STRING,
  age INT,
  city STRING,
  salary DOUBLE
);

-- First load
COPY INTO employees_force
FROM '/Volumes/workspace/default/copy_into_demo/csv/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true');

SELECT 'First load' as stage, COUNT(*) as count FROM employees_force;

-- Force reload the same files (override idempotency)
COPY INTO employees_force
FROM '/Volumes/workspace/default/copy_into_demo/csv/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('force' = 'true');

SELECT 'After force reload' as stage, COUNT(*) as count FROM employees_force;

-- ⚠️ Notice: With 'force' = 'true', data is duplicated!
-- Use 'force' carefully - it bypasses idempotency

-- Common COPY_OPTIONS:
-- 'force' = 'true'        -- Reload already-processed files
-- 'mergeSchema' = 'true'  -- Merge new columns into table schema
-- 'validate' = 'true'     -- Validate but don't load (dry run)

In [0]:
# Incremental loading: Simulating new files arriving over time
# COPY INTO automatically tracks and loads only new files

print("\n=== Incremental Loading Simulation ===")

# Create a new batch of CSV files (simulating new data arrival)
new_csv_data = """id,name,age,city,salary
7,Grace,31,Miami,73000
8,Henry,27,Portland,67000
9,Iris,33,Phoenix,76000"""

dbutils.fs.put("/Volumes/workspace/default/copy_into_demo/csv/employees_3.csv", new_csv_data, overwrite=True)
print("\n✅ Created new file: employees_3.csv")

# Check current count
print("\nBefore loading new file:")
current_count = spark.sql("SELECT COUNT(*) as count FROM employees").collect()[0]['count']
print(f"  Row count: {current_count}")

# Run COPY INTO again - it will load ONLY the new file
print("\nRunning COPY INTO...")
spark.sql("""
COPY INTO employees
FROM '/Volumes/workspace/default/copy_into_demo/csv/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
""")

print("\nAfter loading new file:")
new_count = spark.sql("SELECT COUNT(*) as count FROM employees").collect()[0]['count']
print(f"  Row count: {new_count}")
print(f"  New rows added: {new_count - current_count}")

print("\n✅ COPY INTO automatically detected and loaded only the new file!")
print("This is the key benefit for incremental batch loading.")

# Show all employees
print("\nAll employees:")
spark.sql("SELECT * FROM employees ORDER BY id").show()

In [0]:
%sql
-- Validation mode: Check files without actually loading them
-- Useful for testing before running the actual load

CREATE OR REPLACE TABLE employees_test (
  id INT,
  name STRING,
  age INT,
  city STRING,
  salary DOUBLE
);

-- Validate files without loading (dry run)
COPY INTO employees_test
FROM '/Volumes/workspace/default/copy_into_demo/csv/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('validate' = 'true');

-- Check if any data was loaded
SELECT 
  'After validation' as stage,
  COUNT(*) as row_count 
FROM employees_test;

-- ✅ Row count is 0 - validation doesn't load data
-- Use this to test your COPY INTO configuration before actual loading

In [0]:
%sql
-- View the history of COPY INTO operations
-- This shows which files have been processed

-- Use DESCRIBE HISTORY to see load operations
DESCRIBE HISTORY employees LIMIT 10;

-- For more detailed file-level tracking, query the Delta log
-- This shows exactly which files were loaded in each operation
SELECT 
  version,
  timestamp,
  operation,
  operationParameters.source as source_path,
  operationMetrics.numFiles as files_loaded,
  operationMetrics.numOutputRows as rows_loaded
FROM (DESCRIBE HISTORY employees)
WHERE operation = 'COPY INTO'
ORDER BY version DESC;

In [0]:
# Error handling in COPY INTO: PERMISSIVE, DROPMALFORMED, FAILFAST

print("\n=== Error Handling Modes ===")
print("\nThree modes for handling bad records:")
print("1. PERMISSIVE (default) - Load bad records as NULL values")
print("2. DROPMALFORMED - Skip bad records, load good ones")
print("3. FAILFAST - Fail immediately on first bad record")

# Create a file with some malformed data
bad_csv_data = """id,name,age,city,salary
10,Jack,35,Chicago,80000
11,Kate,invalid_age,LA,75000
12,Leo,28,Seattle,72000"""

dbutils.fs.put("/Volumes/workspace/default/copy_into_demo/csv/bad_data.csv", bad_csv_data, overwrite=True)
print("\n✅ Created file with bad data (invalid age for Kate)")

# Example 1: PERMISSIVE mode (default)
print("\n--- Mode 1: PERMISSIVE (default) ---")
spark.sql("""
CREATE OR REPLACE TABLE employees_permissive (
  id INT,
  name STRING,
  age INT,
  city STRING,
  salary DOUBLE
)
""")

spark.sql("""
COPY INTO employees_permissive
FROM '/Volumes/workspace/default/copy_into_demo/csv/'
FILEFORMAT = CSV
PATTERN = 'bad_data.csv'
FORMAT_OPTIONS (
  'header' = 'true',
  'mode' = 'PERMISSIVE'
)
""")

print("Results with PERMISSIVE mode:")
spark.sql("SELECT * FROM employees_permissive ORDER BY id").show()
print("✅ Bad record loaded with NULL values for unparseable fields")

# Example 2: DROPMALFORMED mode
print("\n--- Mode 2: DROPMALFORMED ---")
spark.sql("""
CREATE OR REPLACE TABLE employees_drop (
  id INT,
  name STRING,
  age INT,
  city STRING,
  salary DOUBLE
)
""")

spark.sql("""
COPY INTO employees_drop
FROM '/Volumes/workspace/default/copy_into_demo/csv/'
FILEFORMAT = CSV
PATTERN = 'bad_data.csv'
FORMAT_OPTIONS (
  'header' = 'true',
  'mode' = 'DROPMALFORMED'
)
""")

print("Results with DROPMALFORMED mode:")
spark.sql("SELECT * FROM employees_drop ORDER BY id").show()
print("✅ Bad record skipped, only valid records loaded")

print("\n--- Mode 3: FAILFAST ---")
print("FAILFAST would fail the entire operation on the first bad record.")
print("Use when data quality is critical and you need strict validation.")

In [0]:
%sql
-- Column mapping: Load only specific columns from source files
-- Useful when source files have more columns than needed

-- Create a table with fewer columns
CREATE OR REPLACE TABLE employees_subset_cols (
  id INT,
  name STRING,
  salary DOUBLE
);

-- COPY INTO will automatically map matching column names
-- Extra columns in source files are ignored
COPY INTO employees_subset_cols
FROM '/Volumes/workspace/default/copy_into_demo/csv/'
FILEFORMAT = CSV
PATTERN = 'employees_1.csv'
FORMAT_OPTIONS ('header' = 'true');

SELECT * FROM employees_subset_cols ORDER BY id;

-- ✅ Only id, name, and salary columns loaded
-- age and city columns from CSV were ignored

-- Note: Column names must match exactly (case-sensitive)
-- COPY INTO does NOT support column renaming or transformations
-- For complex transformations, use:
--   1. Load to staging table with COPY INTO
--   2. Transform with INSERT INTO ... SELECT

In [0]:
# Production pattern: COPY INTO in a scheduled job
# This is the typical use case for COPY INTO

print("\n=== Production Pattern: Scheduled COPY INTO Job ===")
print("\nTypical workflow:")
print("1. Files land in cloud storage (S3, ADLS, GCS)")
print("2. Job runs on schedule (hourly, daily, etc.)")
print("3. COPY INTO loads only new files")
print("4. Job completes and waits for next run")

def incremental_load_job():
    """
    Production-ready incremental load function.
    This would be run on a schedule (e.g., every hour).
    """
    import datetime
    
    print(f"\n[{datetime.datetime.now()}] Starting incremental load...")
    
    try:
        # Run COPY INTO
        result = spark.sql("""
            COPY INTO employees
            FROM '/Volumes/workspace/default/copy_into_demo/csv/'
            FILEFORMAT = CSV
            FORMAT_OPTIONS ('header' = 'true')
        """)
        
        # Get metrics from result
        if result.first():
            num_files = result.first()['num_affected_rows'] if 'num_affected_rows' in result.columns else 0
            print(f"\u2705 Success! Processed files in this run.")
        else:
            print(f"\u2705 Success! No new files to process.")
            
    except Exception as e:
        print(f"❌ Error during load: {str(e)}")
        # In production, you would:
        # - Log to monitoring system
        # - Send alerts
        # - Maybe retry
        raise
    
    print(f"[{datetime.datetime.now()}] Load complete!\n")

# Simulate running the job
print("\nSimulating job execution:")
incremental_load_job()

print("\n=== Recommended Setup ===")
print("\n1. Create a Databricks Job (formerly Workflow)")
print("2. Set schedule: cron expression or simple interval")
print("3. Configure retries and alerting")
print("4. Monitor via job run history")
print("5. Use COPY INTO with appropriate COPY_OPTIONS")
print("\n✅ This pattern handles thousands of files efficiently!")

## COPY INTO vs Auto Loader (cloudFiles)

### When to Use COPY INTO

✅ **Best for:**
* **Small to medium scale** — Hundreds to thousands of files
* **Scheduled batch jobs** — Hourly, daily, weekly loads
* **Simple use cases** — Straightforward file-to-table loading
* **SQL-first workflows** — Pure SQL with no Python/Scala needed
* **Low latency not required** — Batch processing is acceptable

### When to Use Auto Loader

✅ **Best for:**
* **Large scale** — Millions of files, petabytes of data
* **Low latency** — Near real-time processing required
* **Complex transformations** — Need full DataFrame API
* **Schema evolution** — Automatic schema inference and evolution
* **Production streaming** — Continuous incremental processing
* **File notification mode** — Use cloud notifications (S3 events, ADLS EventGrid)

---

## Feature Comparison

| Feature | COPY INTO | Auto Loader |
|---------|-----------|-------------|
| **Syntax** | SQL | Python/Scala (readStream) |
| **File tracking** | Built-in | Checkpoint-based |
| **Idempotency** | ✅ Yes | ✅ Yes |
| **Incremental** | ✅ Yes | ✅ Yes |
| **Scale** | Thousands of files | Millions+ of files |
| **Latency** | Minutes (batch) | Seconds (streaming) |
| **Schema inference** | Basic | Advanced + evolution |
| **Transformations** | Limited | Full DataFrame API |
| **File notifications** | ❌ No | ✅ Yes (S3/ADLS events) |
| **Cloud cost** | Lower (batch) | Higher (streaming) |
| **Complexity** | Low | Medium |
| **Best for** | Scheduled jobs | Production pipelines |

---

## Migration Path

**Start with COPY INTO**, then migrate to Auto Loader when:
* File volume grows beyond thousands
* Latency requirements decrease
* Need advanced schema evolution
* Want to leverage cloud file notifications

**Example Auto Loader equivalent:**

```python
# COPY INTO (SQL)
COPY INTO my_table
FROM '/path/to/files'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')

# Auto Loader equivalent (Python)
df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .load("/path/to/files"))

df.writeStream
    .option("checkpointLocation", "/checkpoint")
    .trigger(availableNow=True)  # Batch-like behavior
    .toTable("my_table")
```

---

## Cost Considerations

**COPY INTO:**
* Runs only when scheduled
* Lower compute costs for batch workloads
* No checkpoint storage needed

**Auto Loader:**
* Can run continuously (higher cost)
* Use `trigger(availableNow=True)` for cost-effective batch mode
* Requires checkpoint storage

---

**✅ Recommendation:** Start with COPY INTO for simplicity, graduate to Auto Loader for scale!

## COPY INTO Best Practices

---

### 1. Table Design

✅ **Create tables explicitly** — Don't rely on schema inference  
✅ **Use Delta tables** — Required for COPY INTO  
✅ **Define constraints** — Add NOT NULL, CHECK constraints  
✅ **Partition appropriately** — For large tables, partition by date  
✅ **Document schema** — Add COMMENT on columns  

```sql
CREATE TABLE my_table (
  id INT NOT NULL,
  event_date DATE NOT NULL,
  value DOUBLE,
  CONSTRAINT pk PRIMARY KEY (id)
)
PARTITIONED BY (event_date)
COMMENT 'Production event table';
```

---

### 2. File Organization

✅ **Consistent naming** — Use predictable file names  
✅ **Date-based folders** — `/year=2026/month=06/day=12/`  
✅ **Moderate file sizes** — 100MB-1GB per file ideal  
✅ **Compress files** — Use gzip, snappy for CSV/JSON  
✅ **Use Parquet when possible** — Better compression and performance  

---

### 3. COPY INTO Configuration

✅ **Use PATTERN for filtering** — Load only relevant files  
✅ **Set appropriate FORMAT_OPTIONS** — Match source file format  
✅ **Choose error mode wisely** — DROPMALFORMED for production  
✅ **Don't use 'force'** — Unless explicitly needed  
✅ **Validate first** — Use 'validate' = 'true' for testing  

```sql
COPY INTO my_table
FROM '/path/to/files'
FILEFORMAT = CSV
PATTERN = '*.csv'
FORMAT_OPTIONS (
  'header' = 'true',
  'mode' = 'DROPMALFORMED',
  'dateFormat' = 'yyyy-MM-dd'
);
```

---

### 4. Scheduling & Automation

✅ **Use Databricks Jobs** — Built-in scheduling and monitoring  
✅ **Set appropriate frequency** — Match data arrival pattern  
✅ **Configure retries** — Handle transient failures  
✅ **Add alerting** — Get notified on failures  
✅ **Monitor run history** — Track success/failure rates  

---

### 5. Monitoring & Troubleshooting

✅ **Check DESCRIBE HISTORY** — See all COPY operations  
✅ **Monitor file counts** — Verify expected files loaded  
✅ **Track row counts** — Compare source vs target  
✅ **Review Delta logs** — Detailed operation metadata  
✅ **Set up data quality checks** — Post-load validation  

```sql
-- Check recent COPY operations
SELECT 
  timestamp,
  operationMetrics.numFiles as files,
  operationMetrics.numOutputRows as rows
FROM (DESCRIBE HISTORY my_table)
WHERE operation = 'COPY INTO'
ORDER BY timestamp DESC
LIMIT 10;
```

---

### 6. Error Handling

✅ **Use DROPMALFORMED in production** — Skip bad records  
✅ **Log dropped records** — Track data quality issues  
✅ **Set up dead letter queue** — Save bad records for review  
✅ **Implement data validation** — Post-load quality checks  
✅ **Alert on anomalies** — Unexpected record counts  

---

### 7. Performance Optimization

✅ **Use Parquet over CSV** — 10-100x faster  
✅ **Compress files** — Reduce network I/O  
✅ **Right-size files** — Not too small, not too large  
✅ **Partition large tables** — Improves query performance  
✅ **OPTIMIZE tables regularly** — Compact small files  

---

### 8. Common Anti-Patterns

❌ **Don't use 'force' routinely** — Breaks idempotency  
❌ **Don't skip validation** — Test configuration first  
❌ **Don't ignore errors** — Monitor and alert  
❌ **Don't mix file formats** — One format per directory  
❌ **Don't overuse FAILFAST** — Too strict for production  
❌ **Don't forget to partition** — Large tables need partitioning  

---

### 9. Migration Checklist

**Moving from manual loads to COPY INTO:**

☐ Create target Delta table with proper schema  
☐ Test COPY INTO with validation mode  
☐ Run initial full load  
☐ Verify data quality and completeness  
☐ Set up scheduled job  
☐ Configure monitoring and alerts  
☐ Document file format and options  
☐ Test failure scenarios  
☐ Implement data quality checks  
☐ Monitor first few runs closely  

---

### 10. When to Graduate to Auto Loader

Consider Auto Loader when:
* Processing > 10,000 files regularly
* Need latency < 5 minutes
* Complex schema evolution required
* Want to use cloud file notifications
* Need advanced monitoring and lineage

---

**✅ Key Takeaway:** COPY INTO is perfect for scheduled batch ingestion at small-to-medium scale. Simple, reliable, and SQL-native!

## COPY INTO - Complete Reference Guide

---

### Full Syntax

```sql
COPY INTO [catalog.]schema.table_name
FROM 'source_path'
FILEFORMAT = file_format
[PATTERN = 'glob_pattern']
[FORMAT_OPTIONS (
  'option1' = 'value1',
  'option2' = 'value2'
)]
[COPY_OPTIONS (
  'option1' = 'value1',
  'option2' = 'value2'
)]
```

---

### Supported File Formats

| Format | FILEFORMAT Value | Common Use |
|--------|------------------|------------|
| CSV | `CSV` | Comma-separated values |
| JSON | `JSON` | JSON records (line-delimited) |
| Parquet | `PARQUET` | Columnar format (recommended) |
| Avro | `AVRO` | Schema evolution support |
| ORC | `ORC` | Optimized Row Columnar |
| Text | `TEXT` | Plain text files |

---

### FORMAT_OPTIONS by File Type

#### CSV Options
```sql
FORMAT_OPTIONS (
  'header' = 'true',              -- First row is header
  'delimiter' = ',',              -- Field delimiter (default: comma)
  'quote' = '"',                  -- Quote character
  'escape' = '\\',                -- Escape character
  'nullValue' = 'NULL',           -- String representing NULL
  'emptyValue' = '',              -- How to treat empty strings
  'dateFormat' = 'yyyy-MM-dd',    -- Date format pattern
  'timestampFormat' = 'yyyy-MM-dd HH:mm:ss',
  'mode' = 'PERMISSIVE',          -- PERMISSIVE, DROPMALFORMED, FAILFAST
  'encoding' = 'UTF-8',           -- Character encoding
  'inferSchema' = 'false',        -- Use table schema instead of inferring
  'columnNameOfCorruptRecord' = '_corrupt_record'
)
```

#### JSON Options
```sql
FORMAT_OPTIONS (
  'multiLine' = 'false',          -- Single-line (true) or multi-line JSON
  'dateFormat' = 'yyyy-MM-dd',
  'timestampFormat' = 'yyyy-MM-dd HH:mm:ss',
  'mode' = 'PERMISSIVE',
  'allowComments' = 'false',      -- Allow Java/C++ style comments
  'allowUnquotedFieldNames' = 'false',
  'allowSingleQuotes' = 'true',   -- Allow single quotes
  'allowNumericLeadingZeros' = 'false',
  'encoding' = 'UTF-8'
)
```

#### Parquet Options
```sql
FORMAT_OPTIONS (
  'mergeSchema' = 'false',        -- Merge schemas from multiple files
  'datetimeRebaseMode' = 'LEGACY' -- Date/time compatibility mode
)
```

#### Avro Options
```sql
FORMAT_OPTIONS (
  'avroSchema' = '{...}',         -- JSON schema string (optional)
  'mode' = 'PERMISSIVE'
)
```

---

### COPY_OPTIONS

| Option | Values | Default | Description |
|--------|--------|---------|-------------|
| `force` | true/false | false | Reload already-processed files |
| `mergeSchema` | true/false | false | Add new columns to table schema |
| `validate` | true/false | false | Dry run - validate without loading |

```sql
COPY_OPTIONS (
  'force' = 'false',              -- Don't reload processed files
  'mergeSchema' = 'false',        -- Don't add new columns
  'validate' = 'false'            -- Actually load data
)
```

---

### Mode Options (Error Handling)

| Mode | Behavior | Use Case |
|------|----------|----------|
| **PERMISSIVE** | Load bad records with NULL values | Development, exploration |
| **DROPMALFORMED** | Skip bad records, load good ones | Production (recommended) |
| **FAILFAST** | Fail on first bad record | Strict validation |

---

### Pattern Matching

Use `PATTERN` to selectively load files:

```sql
-- All files
PATTERN = '*'

-- Specific extension
PATTERN = '*.csv'

-- Prefix match
PATTERN = 'sales_*.json'

-- Date-based
PATTERN = '2026-06-*.parquet'

-- Exact file
PATTERN = 'data.csv'
```

---

### Common Patterns

#### 1. Basic CSV Load
```sql
COPY INTO my_table
FROM '/data/csv/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true');
```

#### 2. JSON with Error Handling
```sql
COPY INTO my_table
FROM '/data/json/'
FILEFORMAT = JSON
FORMAT_OPTIONS ('mode' = 'DROPMALFORMED');
```

#### 3. Parquet with Pattern
```sql
COPY INTO my_table
FROM '/data/parquet/'
FILEFORMAT = PARQUET
PATTERN = '2026-06-*.parquet';
```

#### 4. Validation Only
```sql
COPY INTO my_table
FROM '/data/csv/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('validate' = 'true');
```

#### 5. Force Reload
```sql
COPY INTO my_table
FROM '/data/csv/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('force' = 'true');
```

---

### Monitoring Queries

#### Check Load History
```sql
DESCRIBE HISTORY my_table;
```

#### Get COPY INTO Operations
```sql
SELECT 
  version,
  timestamp,
  operation,
  operationParameters.source,
  operationMetrics.numFiles,
  operationMetrics.numOutputRows
FROM (DESCRIBE HISTORY my_table)
WHERE operation = 'COPY INTO'
ORDER BY version DESC;
```

#### Verify Row Counts
```sql
SELECT COUNT(*) FROM my_table;
```

---

### Limitations

⚠️ **COPY INTO limitations:**

* **No transformations** — Cannot transform data during load
* **No column renaming** — Column names must match exactly
* **Delta tables only** — Target must be Delta table
* **Single source path** — Cannot copy from multiple paths in one statement
* **No streaming** — Batch-only operation
* **File tracking overhead** — Struggles with millions of files
* **No nested field selection** — Limited support for complex types

**For these cases, use:**
* Transformations → Load to staging table, then INSERT INTO ... SELECT
* Millions of files → Auto Loader (cloudFiles)
* Streaming → Structured Streaming or Auto Loader
* Complex transformations → Spark DataFrames

---

### Performance Tips

✅ **File format**: Use Parquet for best performance  
✅ **File size**: 100MB-1GB per file is optimal  
✅ **Compression**: Enable compression (gzip, snappy)  
✅ **Partitioning**: Partition target table by date/category  
✅ **OPTIMIZE**: Run OPTIMIZE regularly to compact files  
✅ **VACUUM**: Clean up old files with VACUUM  
✅ **Pattern matching**: Use PATTERN to limit files processed  

---

### Troubleshooting

#### Issue: Files not loading
* Check file path and permissions
* Verify PATTERN matches files
* Look for schema mismatches
* Check DESCRIBE HISTORY for errors

#### Issue: Duplicate data
* Don't use 'force' = 'true' unless needed
* Verify idempotency is working
* Check for file renames (seen as new files)

#### Issue: Schema errors
* Ensure table schema matches file schema
* Check date/timestamp formats
* Verify data types are compatible
* Use 'mode' = 'PERMISSIVE' for debugging

#### Issue: Performance problems
* Use Parquet instead of CSV/JSON
* Check file sizes (too small/large)
* Consider partitioning
* Migrate to Auto Loader for large scale

---

### Python/Scala Equivalent

**SQL:**
```sql
COPY INTO my_table
FROM '/data/csv/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true');
```

**Python (using spark.sql):**
```python
spark.sql("""
  COPY INTO my_table
  FROM '/data/csv/'
  FILEFORMAT = CSV
  FORMAT_OPTIONS ('header' = 'true')
""")
```

**No native PySpark DataFrame API for COPY INTO** — use `spark.sql()` or migrate to Auto Loader.

---

### Migration to Auto Loader

**When to migrate:**
* File count > 10,000
* Need streaming capabilities
* Want schema evolution
* Need file notifications

**COPY INTO:**
```sql
COPY INTO my_table
FROM '/data/csv/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true');
```

**Auto Loader equivalent:**
```python
df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true")
  .load("/data/csv/"))

df.writeStream
  .option("checkpointLocation", "/checkpoint")
  .trigger(availableNow=True)  # Batch mode
  .toTable("my_table")
```

---

### Additional Resources

* **Databricks Documentation**: Search for "COPY INTO"
* **Auto Loader Guide**: For large-scale ingestion
* **Delta Lake**: ACID transactions and time travel
* **Lakeflow Pipelines**: Managed ETL pipelines
* **Unity Catalog**: Data governance and lineage

---

**✅ Quick Reference:**
* **Target**: Delta tables only
* **Idempotent**: Yes (tracks files)
* **Incremental**: Yes (loads only new files)
* **Scale**: Thousands of files
* **Best for**: Scheduled batch ingestion
* **Alternative**: Auto Loader for large scale